# Tutorial 20: HLLSet Disambiguation

This tutorial demonstrates how to use the HLLSet disambiguation module
to find matching TokenLUT entries using zero-copy position matching.

## Key Concepts

1. **Position-based matching**: An HLLSet encodes tokens as (register, zeros) positions.
   Disambiguation finds all LUT entries that match these positions.

2. **Zero-copy in Rust**: The matching happens in the Rust module with direct
   access to the Roaring bitmap - no data is copied to Python.

3. **Streaming output**: Matches are written immediately to a Redis Stream,
   releasing Redis for other operations.

4. **Collision-aware schema**: LUT entries use JSON arrays to store all
   tokens that hash to the same 64-bit value:
   - `first_tokens`: JSON array of first tokens (for triangulation)
   - `tokens`: JSON array of n-gram token arrays

5. **Layer filtering**: Filter by layer (0=unigram, 1=bigram, etc.) to focus
   on specific n-gram types.

In [3]:
import redis
import sys
sys.path.insert(0, '..')

from core.hllset_disambiguate import (
    HLLSetDisambiguator,
    DisambiguationPipeline,
    disambiguate,
    disambiguate_stream,
    Candidate
)

# Connect to Redis
r = redis.Redis(host='127.0.0.1', port=6379, decode_responses=False)

## 1. Setup: Create HLLSet and TokenLUT entries

First, let's create an HLLSet from some tokens and corresponding LUT entries.

In [4]:
# Create an HLLSet from tokens
tokens = ["hello", "world", "this", "is", "a", "test"]
hllset_key = r.execute_command("HLLSET.CREATE", *tokens)
print(f"Created HLLSet: {hllset_key.decode()}")

# Get the positions
positions = r.execute_command("HLLSET.POSITIONS", hllset_key)
print(f"\nPositions (reg, zeros):")
for i in range(0, len(positions), 2):
    print(f"  ({positions[i]}, {positions[i+1]})")

Created HLLSet: hllset:068fd8ba8b85b449e07391d4047cd972d1f17924

Positions (reg, zeros):
  (137, 1)
  (234, 0)
  (413, 3)
  (753, 0)
  (770, 1)
  (824, 0)


In [ ]:
import json

# Create LUT entries using the NEW collision-aware schema
# In a real scenario, these would be created by the TokenLUT session builder

lut_prefix = "tokenlut:demo:entry:"

# Convert positions to list of tuples
pos_list = [(positions[i], positions[i+1]) for i in range(0, len(positions), 2)]

# Create matching LUT entries with JSON arrays
# Schema: first_tokens=["token"], tokens=[] for unigrams
for idx, (token, (reg, zeros)) in enumerate(zip(tokens, pos_list)):
    # Use hash as key (simulated - in real use, use hash_full from hash function)
    entry_key = f"{lut_prefix}{idx:04d}"
    r.hset(entry_key, mapping={
        "reg": str(reg),
        "zeros": str(zeros),
        "hash_full": str(idx),  # Simulated hash
        "layer": "0",  # Unigram
        "first_tokens": json.dumps([token]),  # JSON array
        "tokens": json.dumps([])  # Empty for unigrams
    })
    print(f"Created {entry_key}: first_tokens={[token]} at ({reg}, {zeros})")

# Also create some bigram entries (layer=1)
# Schema: first_tokens=["first"], tokens=[["first", "second"]]
bigrams = [("hello", "world"), ("this", "is")]
for idx, (first, second) in enumerate(bigrams):
    # Create a position for the bigram (hash of "first:second")
    bigram_key = r.execute_command("HLLSET.CREATE", f"{first}:{second}")
    bigram_pos = r.execute_command("HLLSET.POSITIONS", bigram_key)
    
    entry_key = f"{lut_prefix}bigram:{idx:04d}"
    r.hset(entry_key, mapping={
        "reg": str(bigram_pos[0]),
        "zeros": str(bigram_pos[1]),
        "hash_full": str(1000 + idx),  # Simulated hash
        "layer": "1",  # Bigram
        "first_tokens": json.dumps([first]),  # JSON array of first tokens
        "tokens": json.dumps([[first, second]])  # JSON array of token arrays
    })
    print(f"Created bigram {entry_key}: first_tokens={[first]}, tokens={[[first, second]]}")

# Add a non-matching entry to verify filtering works
r.hset(f"{lut_prefix}nomatch:0001", mapping={
    "reg": "999",
    "zeros": "31",
    "hash_full": "9999",
    "layer": "0",
    "first_tokens": json.dumps(["nomatch"]),
    "tokens": json.dumps([])
})
print("\nCreated non-matching entry (should be filtered out)")

Created tokenlut:demo:entry:0000: hello at (137, 1)
Created tokenlut:demo:entry:0001: world at (234, 0)
Created tokenlut:demo:entry:0002: this at (413, 3)
Created tokenlut:demo:entry:0003: is at (753, 0)
Created tokenlut:demo:entry:0004: a at (770, 1)
Created tokenlut:demo:entry:0005: test at (824, 0)
Created bigram tokenlut:demo:entry:bigram:0000: hello -> world
Created bigram tokenlut:demo:entry:bigram:0001: this -> is

Created non-matching entry (should be filtered out)


## 2. Basic Disambiguation

Use `HLLSET.CANDIDATES` to find all LUT entries matching the HLLSet positions.

In [6]:
# Simple disambiguation using the convenience function
candidates = disambiguate(r, hllset_key.decode(), lut_prefix)

print(f"Found {len(candidates)} candidates:\n")
for c in candidates:
    print(f"  Token: {c.token}")
    print(f"    Key: {c.key}")
    print(f"    Position: ({c.reg}, {c.zeros})")
    print(f"    Layer: {c.layer} ({'unigram' if c.layer == 0 else 'bigram'})")
    if c.first_token:
        print(f"    First Token: {c.first_token}")
    print()

Found 6 candidates:

  Token: a
    Key: tokenlut:demo:entry:0004
    Position: (770, 1)
    Layer: 0 (unigram)

  Token: this
    Key: tokenlut:demo:entry:0002
    Position: (413, 3)
    Layer: 0 (unigram)

  Token: test
    Key: tokenlut:demo:entry:0005
    Position: (824, 0)
    Layer: 0 (unigram)

  Token: is
    Key: tokenlut:demo:entry:0003
    Position: (753, 0)
    Layer: 0 (unigram)

  Token: world
    Key: tokenlut:demo:entry:0001
    Position: (234, 0)
    Layer: 0 (unigram)

  Token: hello
    Key: tokenlut:demo:entry:0000
    Position: (137, 1)
    Layer: 0 (unigram)



## 3. Layer Filtering

Filter candidates by layer to get only unigrams or only bigrams.

In [7]:
# Get only unigrams (layer=0)
unigrams = disambiguate(r, hllset_key.decode(), lut_prefix, layer=0)
print(f"Unigrams ({len(unigrams)}):")
for c in unigrams:
    print(f"  {c.token}")

Unigrams (6):
  a
  this
  test
  is
  world
  hello


## 4. Streaming Output

For large-scale disambiguation, stream results to a Redis Stream.
This releases Redis immediately after writing each match.

In [8]:
# Stream results to a Redis Stream
stream_key = "disamb:stream:demo"
r.delete(stream_key)  # Clear any existing stream

count = disambiguate_stream(r, hllset_key.decode(), lut_prefix, stream_key)
print(f"Streamed {count} candidates to {stream_key}")

# Read from the stream
entries = r.xrange(stream_key, "-", "+")
print(f"\nStream entries:")
for msg_id, fields in entries:
    print(f"  ID: {msg_id.decode()}")
    for k, v in fields.items():
        print(f"    {k.decode()}: {v.decode()}")
    print()

Streamed 6 candidates to disamb:stream:demo

Stream entries:
  ID: 1776272624828-0
    key: tokenlut:demo:entry:0004
    reg: 770
    zeros: 1
    layer: 0
    token: a
    first_token: 

  ID: 1776272624828-1
    key: tokenlut:demo:entry:0002
    reg: 413
    zeros: 3
    layer: 0
    token: this
    first_token: 

  ID: 1776272624828-2
    key: tokenlut:demo:entry:0005
    reg: 824
    zeros: 0
    layer: 0
    token: test
    first_token: 

  ID: 1776272624828-3
    key: tokenlut:demo:entry:0003
    reg: 753
    zeros: 0
    layer: 0
    token: is
    first_token: 

  ID: 1776272624828-4
    key: tokenlut:demo:entry:0001
    reg: 234
    zeros: 0
    layer: 0
    token: world
    first_token: 

  ID: 1776272624828-5
    key: tokenlut:demo:entry:0000
    reg: 137
    zeros: 1
    layer: 0
    token: hello
    first_token: 



## 5. Using the HLLSetDisambiguator Class

For more control, use the `HLLSetDisambiguator` class.

In [9]:
disamb = HLLSetDisambiguator(r)

# Create a position index (sorted set)
index_key = "disamb:index:demo"
r.delete(index_key)
count = disamb.create_position_index(hllset_key.decode(), index_key)
print(f"Created position index with {count} entries")

# View the index (positions as scores)
print("\nPosition index (member: score):")
for member, score in r.zrange(index_key, 0, -1, withscores=True):
    print(f"  {member.decode()}: {score}")

Created position index with 6 entries

Position index (member: score):
  137:1: 4385.0
  234:0: 7488.0
  413:3: 13219.0
  753:0: 24096.0
  770:1: 24641.0
  824:0: 26368.0


## 6. Disambiguation Pipeline

For complex disambiguation, use the `DisambiguationPipeline` class
to compose multiple filtering stages.

In [10]:
# Create a multi-stage pipeline
pipeline = DisambiguationPipeline(r)

# Stage 1: Get all unigrams
pipeline.add_stage("unigrams", layer=0)

# Stage 2: Get all candidates with a custom filter
def token_length_filter(c: Candidate) -> bool:
    """Keep only tokens with length > 2"""
    return len(c.token) > 2

pipeline.add_stage("long_tokens", filter_fn=token_length_filter)

# Run the pipeline
results = pipeline.run(hllset_key.decode(), lut_prefix)

print("Pipeline results:")
for stage_name, candidates in results.items():
    print(f"\n{stage_name}:")
    for c in candidates:
        print(f"  - {c.token}")

Pipeline results:

unigrams:
  - a
  - this
  - test
  - is
  - world
  - hello

long_tokens:
  - this
  - test
  - world
  - hello


## 7. Performance: SCANMATCH for Large Datasets

For very large LUT prefixes, use `HLLSET.SCANMATCH` which handles
proper cursor iteration internally.

In [11]:
# Use SCANMATCH for large-scale disambiguation
scan_stream = "disamb:scanstream:demo"
r.delete(scan_stream)

total = disamb.scan_match(
    hllset_key.decode(),
    lut_prefix,
    scan_stream,
    batch_size=100  # Process 100 keys per iteration
)

print(f"SCANMATCH found {total} matches")
print(f"Stream length: {r.xlen(scan_stream)}")

SCANMATCH found 6 matches
Stream length: 6


## 8. Consuming Stream Results

Use the `consume_stream` method to iterate over stream entries as Candidate objects.

In [12]:
# Consume stream as Candidate objects
print("Consuming stream:")
for candidate in disamb.consume_stream(scan_stream):
    print(f"  {candidate.token} at ({candidate.reg}, {candidate.zeros})")

Consuming stream:
  a at (770, 1)
  this at (413, 3)
  test at (824, 0)
  is at (753, 0)
  world at (234, 0)
  hello at (137, 1)


## Summary

The HLLSet disambiguation module provides:

1. **HLLSET.CANDIDATES** - Find matching LUT entries with optional layer filter
2. **HLLSET.SCANMATCH** - Full cursor-based scan for large datasets
3. **HLLSET.POSINDEX** - Create a sorted set index of positions

### New Collision-Aware Schema

LUT entries use JSON arrays to handle hash collisions:

| Layer | `first_tokens` | `tokens` |
|-------|---------------|----------|
| 0 (unigram) | `["apple", "app"]` | `[]` |
| 1 (bigram) | `["quick", "quest"]` | `[["quick","brown"],["quest","ion"]]` |
| 2 (trigram) | `["the"]` | `[["the","quick","brown"]]` |

Key benefits:
- **Content-addressable keying**: `key = prefix + hash_full` (64-bit hash)
- **Collision handling**: Multiple tokens with same hash are stored together
- **Fast register filtering**: RediSearch numeric query on `reg` field
- **Triangulation support**: `first_tokens` array for n-gram constraints

In [ ]:
# Cleanup (optional)
# r.delete(hllset_key)
# for key in r.scan_iter(f"{lut_prefix}*"):
#     r.delete(key)
# r.delete(stream_key, scan_stream, index_key)